In [1]:
import requests
import pandas as pd
import kagglehub
from recognition_evaluators import evaluate_tm_df
from util import dictcount

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kzvau\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
BASE_URL = "https://deepcode.ci.nsu.ru"
api_key = "sk-d03991b7df2a4bd0b306fe9dbb2d6389"
headers = {"Authorization": f"Bearer {api_key}"}

MODEL_ID = "Qwen3.8-27B"
print("Model:", MODEL_ID)


Model: Qwen3.8-27B


In [3]:
multiclass_dataset_path = kagglehub.dataset_download("sagarikashreevastava/cognitive-distortion-detetction-dataset")
multiclass_dataset_file_path = multiclass_dataset_path + "/Annotated_data.csv"
df = pd.read_csv(multiclass_dataset_file_path)
df = df.drop("Id_Number", axis=1)
df

,Patient Question,Distorted part,Dominant Distortion,Secondary Distortion (Optional)
0,"Hello, I have a beautiful,smart,outgoing and a...",The voice are always fimilar (someone she know...,Personalization,NaN
1,Since I was about 16 years old I’ve had these ...,I feel trapped inside my disgusting self and l...,Labeling,Emotional Reasoning
2,So I’ve been dating on and off this guy for a...,NaN,No Distortion,NaN
3,My parents got divorced in 2004. My mother has...,NaN,No Distortion,NaN
4,I don’t really know how to explain the situati...,I refused to go because I didn’t know if it wa...,Fortune-telling,Emotional Reasoning
...,...,...,...,...
2525,I’m a 21 year old female. I spent most of my l...,NaN,No Distortion,NaN
2526,I am 21 female and have not had any friends fo...,Now I am at university my peers around me all ...,Overgeneralization,NaN
2527,From the U.S.: My brother is 19 years old and ...,He claims he’s severely depressed and has outb...,Mental filter,Mind Reading
2528,From the U.S.: I am a 21 year old woman who ha...,NaN,No Distortion,NaN


In [4]:
all_dists = {}
for _, row in df.iterrows():
    primary_distortion = row.iloc[2]
    secondary_distortion = row.iloc[3] if pd.notna(row.iloc[3]) else None
    dictcount(all_dists, primary_distortion)
    if secondary_distortion is not None:
        dictcount(all_dists, secondary_distortion)
del all_dists["No Distortion"]

split_shifts = [0, 1, 2]
validation_fraction = 5
labels = sorted(all_dists)
all_dists


{'Personalization': 202,
 'Labeling': 203,
 'Emotional Reasoning': 169,
 'Fortune-telling': 210,
 'Magnification': 245,
 'Mind Reading': 295,
 'All-or-nothing thinking': 126,
 'Overgeneralization': 277,
 'Mental filter': 151,
 'Should statements': 135}

In [5]:
def evaluate_specific_model(model, text, dists, debug=False):
    global last_response
    query = (
        f'You are professional psycho-therapist experienced in cognitive-behavioral therapy. '
        f'You can label texts witn none or some of cognitive distortoions, represented by the following labels: {str(dists)}. '
        f'When labeling, return only JSON array in square brackets of strings in double quotes representing the labels. '
        f'Label this text for presense or absence of any cognitive distortions fom given list: "{text}"'
    )
    if debug:
        print(query)
    response = requests.post(
        BASE_URL + "/api/chat/completions",
        headers=headers,
        json={
            "model": model,
            "messages": [{"role": "user", "content": query}],
            "temperature": 0.0,
            "chat_template_kwargs": {"enable_thinking": False},
            "max_tokens": 4096,
            "stream": False
        },
        timeout=300
    )
    response.raise_for_status()
    last_response = response.json()
    return last_response["choices"][0]["message"]["content"]


def llm_evaluator_top1(all_metrics, model, text, threshold, top=1):
    rep = evaluate_specific_model(model, text, dists=all_metrics, debug=False)
    rep_list = []
    for metric in all_metrics:
        if metric in rep:
            rep_list.append(metric)
    return {value: 1.0 for index, value in enumerate(rep_list[:top])}

In [6]:
predictions_by_split = {split_shift: [] for split_shift in split_shifts}


In [7]:
for split_shift in split_shifts:
    test_df = df[(df.index + split_shift) % validation_fraction == 0]
    predictions = predictions_by_split[split_shift]
    print("Split shift:", split_shift, "Rows:", len(test_df))

    for index, row in test_df.iloc[len(predictions):].iterrows():
        text = row.iloc[1] if pd.notna(row.iloc[1]) else row.iloc[0]
        predictions.append(llm_evaluator_top1(all_dists, MODEL_ID, text, None))


Split shift: 0 Rows: 506
Split shift: 1 Rows: 506
Split shift: 2 Rows: 506


In [8]:
def stored_evaluator(all_metrics, predictions_iter, text, threshold, min_count):
    return next(predictions_iter)

all_eval_results = []
results = []
metrics = []
summaries = []

for split_shift in split_shifts:
    test_df = df[(df.index + split_shift) % validation_fraction == 0]
    predictions = predictions_by_split[split_shift]
    eval_result = evaluate_tm_df(
        test_df, iter(predictions), stored_evaluator, None,
        all_metrics=all_dists, encode_spaces=False, debug=False
    )
    all_eval_results.append(eval_result)
    precision, recall, f1, accuracy = eval_result

    for (index, row), prediction in zip(test_df.iterrows(), predictions):
        labels_for_row = []
        if row.iloc[2] != "No Distortion":
            labels_for_row.append(row.iloc[2])
        if pd.notna(row.iloc[3]) and row.iloc[3] != "No Distortion":
            labels_for_row.append(row.iloc[3])
        results.append({
            "split_shift": split_shift,
            "source_index": index,
            "text": row.iloc[1] if pd.notna(row.iloc[1]) else row.iloc[0],
            "true_labels": labels_for_row,
            "predicted_labels": list(prediction)
        })

    for label in labels:
        metrics.append({
            "split_shift": split_shift,
            "label": label,
            "precision": precision[label],
            "recall": recall[label],
            "f1": f1[label],
            "accuracy": accuracy[label]
        })
    summaries.append({
        "model": MODEL_ID,
        "split_shift": split_shift,
        "test_rows": len(test_df),
        "macro_f1": sum(f1.values()) / len(f1)
    })

results_df = pd.DataFrame(results)
metrics_df = pd.DataFrame(metrics)
summary_df = pd.DataFrame(summaries)


In [9]:
F1s = [sum(run[2].values()) / len(run[2]) for run in all_eval_results]
F1_macro = sum(F1s) / len(F1s)
F1_minmax = (min(F1s), max(F1s))

display(results_df)
display(metrics_df)
display(summary_df)
print("Mean macro F1:", F1_macro)
print("Macro F1 range:", F1_minmax)
print("zero_shot_f1s =", F1s)


,split_shift,source_index,text,true_labels,predicted_labels
0,0,0,The voice are always fimilar (someone she know...,[Personalization],[]
1,0,5,Hello. I have been friend with a guy since gra...,[],[Mind Reading]
2,0,10,"I thought that he displayed traits of honor, l...",[Labeling],[]
3,0,15,"I started going to therapy in December, after ...",[],[]
4,0,20,During this time I was recruited to many great...,[Fortune-telling],[]
...,...,...,...,...,...
1513,2,2508,From the U.S. — I’m 15. So ever since a couple...,[],[]
1514,2,2513,I am very worried about my older brother. In t...,[],[]
1515,2,2518,"Since I can remember, I’ve always thought of m...","[Labeling, Overgeneralization]",[Fortune-telling]
1516,2,2523,"However, I am not happy, at the least only hal...","[Labeling, All-or-nothing thinking]",[]


,split_shift,label,precision,recall,f1,accuracy
0,0,All-or-nothing thinking,0.185185,1.0,0.312500,0.009881
1,0,Emotional Reasoning,0.000000,0.0,0.000000,0.000000
2,0,Fortune-telling,0.388060,1.0,0.559140,0.051383
3,0,Labeling,0.421053,1.0,0.592593,0.047431
4,0,Magnification,0.727273,1.0,0.842105,0.015810
5,0,Mental filter,0.000000,0.0,0.000000,0.000000
6,0,Mind Reading,0.500000,1.0,0.666667,0.035573
7,0,Overgeneralization,0.514286,1.0,0.679245,0.035573
8,0,Personalization,0.318182,1.0,0.482759,0.013834
9,0,Should statements,0.733333,1.0,0.846154,0.021739


,model,split_shift,test_rows,macro_f1
0,Qwen3.8-27B,0,506,0.498116
1,Qwen3.8-27B,1,506,0.465688
2,Qwen3.8-27B,2,506,0.587507


Mean macro F1: 0.5171038613834863
Macro F1 range: (0.4656882712015326, 0.5875071072263505)
zero_shot_f1s = [0.498116205722576, 0.4656882712015326, 0.5875071072263505]


In [10]:
zero_shot_f1_by_label = {
    label: [run[2][label] for run in all_eval_results]
    for label in labels
}
print("zero_shot_f1_by_label =", zero_shot_f1_by_label)


zero_shot_f1_by_label = {'All-or-nothing thinking': [0.3125, 0.2758620689655173, 0.3448275862068966], 'Emotional Reasoning': [0, 0, 0.33333333333333337], 'Fortune-telling': [0.5591397849462366, 0.5420560747663552, 0.5054945054945055], 'Labeling': [0.5925925925925926, 0.7777777777777778, 0.5070422535211268], 'Magnification': [0.8421052631578948, 0.33333333333333337, 0.5454545454545454], 'Mental filter': [0, 0.25, 0.5714285714285715], 'Mind Reading': [0.6666666666666666, 0.7529411764705882, 0.7333333333333334], 'Overgeneralization': [0.6792452830188679, 0.56, 0.6774193548387097], 'Personalization': [0.4827586206896552, 0.631578947368421, 0.7234042553191489], 'Should statements': [0.846153846153846, 0.5333333333333333, 0.9333333333333333]}
